In [5]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [6]:
RANDOM_SEED = 42

LABEL_TO_ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

LABEL_ORDER = ["negative", "neutral", "positive"]

# Find the project root folder
project_root = next(
    folder
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data").exists()
)

processed_dir = project_root / "data" / "processed"

phrasebank_clean = pd.read_csv(
    processed_dir / "financial_phrasebank_50agree_clean.csv"
)

print("Loaded dataset shape:", phrasebank_clean.shape)
display(phrasebank_clean.head())

Loaded dataset shape: (4840, 2)


,sentence,label
0,"According to Gran , the company has no plans t...",neutral
1,Technopolis plans to develop in stages an area...,neutral
2,The international electronic industry company ...,negative
3,With the new production plant the company woul...,positive
4,According to the company 's updated strategy f...,positive


In [7]:
label_summary = pd.DataFrame(
    {
        "count": (
            phrasebank_clean["label"]
            .value_counts()
            .reindex(LABEL_ORDER)
        ),
        "percentage": (
            phrasebank_clean["label"]
            .value_counts(normalize=True)
            .reindex(LABEL_ORDER)
            .mul(100)
            .round(2)
        )
    }
)

label_summary

,count,percentage
label,,
negative,604,12.48
neutral,2873,59.36
positive,1363,28.16


In [8]:
duplicate_summary = (
    phrasebank_clean
    .groupby("sentence")["label"]
    .agg(
        occurrences="size",
        unique_label_count="nunique"
    )
)

repeated_sentences = duplicate_summary[
    duplicate_summary["occurrences"] > 1
]

conflicting_sentences = repeated_sentences[
    repeated_sentences["unique_label_count"] > 1
]

print("Repeated sentences:", len(repeated_sentences))
print(
    "Repeated sentences with conflicting labels:",
    len(conflicting_sentences)
)

Repeated sentences: 2
Repeated sentences with conflicting labels: 2


In [9]:
# Remove texts whose repeated versions have different labels.
# Their target label is ambiguous.
conflicting_texts = conflicting_sentences.index

model_df = phrasebank_clean[
    ~phrasebank_clean["sentence"].isin(conflicting_texts)
].copy()

# Keep one copy of statements repeated with the same label.
model_df = (
    model_df
    .drop_duplicates(subset="sentence", keep="first")
    .reset_index(drop=True)
)

# Create numeric labels for DistilBERT later.
model_df["label_id"] = model_df["label"].map(LABEL_TO_ID)

print("Original dataset size:", len(phrasebank_clean))
print("Model-ready dataset size:", len(model_df))

model_df["label"].value_counts()

Original dataset size: 4840
Model-ready dataset size: 4836


label
neutral     2871
positive    1361
negative     604
Name: count, dtype: int64

In [10]:
# 70% training data; the other 30% will become validation and test data.
train_df, temporary_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["label"],
    random_state=RANDOM_SEED
)

# Split the remaining 30% into 15% validation and 15% test data.
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df["label"],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

Training examples: 3385
Validation examples: 725
Test examples: 726


In [11]:
assert set(train_df["sentence"]).isdisjoint(validation_df["sentence"])
assert set(train_df["sentence"]).isdisjoint(test_df["sentence"])
assert set(validation_df["sentence"]).isdisjoint(test_df["sentence"])

print("No statement appears in more than one split.")

No statement appears in more than one split.


In [12]:
split_balance = pd.DataFrame(
    {
        "full_dataset": (
            model_df["label"]
            .value_counts(normalize=True)
            .reindex(LABEL_ORDER)
            .mul(100)
        ),
        "train": (
            train_df["label"]
            .value_counts(normalize=True)
            .reindex(LABEL_ORDER)
            .mul(100)
        ),
        "validation": (
            validation_df["label"]
            .value_counts(normalize=True)
            .reindex(LABEL_ORDER)
            .mul(100)
        ),
        "test": (
            test_df["label"]
            .value_counts(normalize=True)
            .reindex(LABEL_ORDER)
            .mul(100)
        )
    }
).round(2)

split_balance

,full_dataset,train,validation,test
label,,,,
negative,12.49,12.50,12.41,12.53
neutral,59.37,59.35,59.45,59.37
positive,28.14,28.15,28.14,28.10


In [13]:
train_df.to_csv(processed_dir / "train.csv", index=False)
validation_df.to_csv(processed_dir / "validation.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Saved:")
print(processed_dir / "train.csv")
print(processed_dir / "validation.csv")
print(processed_dir / "test.csv")

Saved:
c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\data\processed\train.csv
c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\data\processed\validation.csv
c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\data\processed\test.csv
